
# (재현) Nature Communications 맥주 논문 ML 학습 파이프라인 — Excel 원본 기반

이 노트북은 Zenodo에 공개된 **"Jupyter Notebook for training machine learning models.ipynb"**의
모델/파라미터/데이터 split 방식을 **최대한 그대로** 따르되,  
입력 데이터를 **Excel(`Supplemental Files and Figure source files.xlsx`)**에서 직접 읽어
저자 노트북이 기대하는 CSV(`Supplemental File S1.csv`, `Supplemental File S4.csv`)를 **런 폴더에 자동 생성**한 뒤
동일한 코드를 실행할 수 있게 만든 재현용 버전입니다.

---

## 재현 범위(중요)
- train/test split:
  - 대부분 모델: `test_size=0.30`, `random_state=0`, `stratify=tasting_category_fine`
  - MLP(신경망): 저자 노트북에서 **random_state=1**로 별도 split(성능 이슈 언급)
- X 전처리:
  - `generate_X(..., impute=True)` (mean impute)
  - `StandardScaler().fit(X_train)` 후 X_train/X_test transform (대부분 모델)
  - Linear/Lasso는 추가로 **1차 interaction term(PolynomialFeatures, interaction_only=True)** 생성
- 모델 라인업(10개):
  - Linear, Lasso, PLS, AdaBoost, GradientBoosting, RandomForest, ExtraTrees, XGBoost, SVR, MLP(Optuna)

> ⚠️ 실행량이 매우 큽니다.  
> 특히 SVR GridSearch(거대한 그리드)와 MLP Optuna(기본 3000 trials × 50 타깃)는 장시간이 걸릴 수 있습니다.  
> 아래 설정 셀의 `RUN_*` 토글로 필요한 섹션만 실행할 수 있게 해두었습니다.

---

## 출력
사용자가 지정한 폴더 아래에 **RUN_TAG(현재시간)** 폴더를 만들고, 저자 노트북과 동일한 폴더 구조로 저장합니다:

- `models/`
- `Model_performance/` (저자 노트북에서 주로 사용)
- `Model_Performance/` (MLP 저장 시 오타로 사용됨 → 둘 다 생성)
- `Optimal_settings/`
- `CV_results/`

즉, 파일명은 저자 노트북과 동일하지만 **경로가 시간별로 분리**되어 재현/비교가 쉽습니다.


In [ ]:

import os
import datetime as dt
import numpy as np
import pandas as pd

from numpy.random import randint

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures

from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LassoCV, MultiTaskLassoCV
from sklearn.cross_decomposition import PLSRegression

from sklearn.multioutput import MultiOutputRegressor

from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import AdaBoostRegressor

# --- optional deps ---
try:
    from xgboost import XGBRegressor
except Exception as e:
    XGBRegressor = None
    print("⚠️ xgboost import 실패:", repr(e))

try:
    import optuna
except Exception as e:
    optuna = None
    print("⚠️ optuna import 실패:", repr(e))

import pickle
import joblib


In [ ]:

# =========================================================
# 0) 사용자 설정(경로/출력/토글)                              # ★ 중요
# =========================================================

# --- 데이터 경로 ---
DATA_XLSX_PATH = r"/home/a202192020/맥주데이터실험/data/Supplemental Files and Figure source files.xlsx"
FALLBACK_XLSX_PATH = r"/mnt/data/Supplemental Files and Figure source files.xlsx"

# --- Excel sheet names (Zenodo 기준) ---
CHEM_SHEET = "Supplementary File S1"
SENS_SHEET = "Supplementary File S4"

# --- 출력 루트 폴더 ---
OUT_ROOT_DIR = r"/home/a202192020/맥주데이터실험/pca/0220/output"

# --- 실행 토글(필요한 것만 True로) ---
RUN_LINEAR_LASSO = True
RUN_PLS          = True
RUN_ADABOOST     = True
RUN_GRADBOOST    = True
RUN_RF           = True
RUN_EXTRATREES   = True
RUN_XGBOOST      = True   # xgboost 설치 필요
RUN_SVR          = False  # 기본 False 권장(그리드가 너무 큼)
RUN_MLP_OPTUNA   = False  # 기본 False 권장(optuna + 장시간)

# --- 병렬수(저자 노트북은 n_jobs=8 사용) ---
N_JOBS = 8

# --- MLP Optuna trials (저자 노트북은 3000) ---
MLP_N_TRIALS = 3000

# --- Run Tag (현재시간) ---
RUN_TAG = dt.datetime.now().strftime("%Y%m%d_%H%M%S")

# --- run 폴더 ---
RUN_DIR = os.path.join(OUT_ROOT_DIR, RUN_TAG)
os.makedirs(RUN_DIR, exist_ok=True)

# 저자 노트북과 동일한 상대경로 저장을 위해 working dir 이동
os.chdir(RUN_DIR)

# 결과 폴더 생성(저자 코드 + init_modeling 요구)
for path in ["models", "Model_performance", "Model_Performance", "Optimal_settings", "CV_results"]:
    os.makedirs(path, exist_ok=True)

np.random.seed(42)

print("RUN_DIR:", RUN_DIR)
print("RUN_TAG:", RUN_TAG)


In [ ]:

# =========================================================
# 1) init_modeling.py 핵심 함수(저자 Zenodo 버전과 동일 로직)  # ★ 재현성
# =========================================================
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.impute import SimpleImputer as Imputer

def generate_X(compounds, impute=False):
    '''
    Zenodo init_modeling.generate_X 로직을 그대로 가져옴.
    - beer_id를 index로 사용
    - 기술/설명 컬럼 drop
    - impute=True면 mean impute
    '''
    index_ = compounds.beer_id
    col_drop = ['date','beer_id', 'ABV','Barcode', 'beer', 'brewery', 'tasting_category']
    X = compounds.copy()
    for col in col_drop:
        if col in X.columns:
            X = X.drop(col, axis=1)
    if 'CO2-PSI.' in X.columns:
        X = X.drop(['CO2-PSI.'], axis=1)

    if impute is True:
        imputer = Imputer(strategy='mean')
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=index_)
    else:
        X.fillna(value=0, axis=1, inplace=True)

    X.set_index(index_, inplace=True)
    return X

def regressor_performance(X_train, Y_train, X_test, Y_test, model, toprint=True):
    y_pred = model.predict(X_train)
    mse_train = mean_squared_error(Y_train, y_pred)
    r2_train = r2_score(Y_train, y_pred)

    y_pred = model.predict(X_test)
    mse_test = mean_squared_error(Y_test, y_pred)
    r2_test = r2_score(Y_test, y_pred)

    if toprint:
        print('Training set performance:')
        print('Mean squared error: ' + str(mse_train))
        print('Coefficient of determination (R^2): ' + str(r2_train))
        print()
        print('Test set performance:')
        print('Mean squared error: ' + str(mse_test))
        print('Coefficient of determination (R^2): ' + str(r2_test))
        print()

    return mse_train, r2_train, mse_test, r2_test

def pickle_cv_results(file_name, cv_results):
    fileObject = open(os.path.join('CV_results', file_name+'_CV_results.pkl'),'wb')
    pickle.dump(cv_results, fileObject)
    fileObject.close()
    cv_results.to_csv(os.path.join('CV_results', file_name+'_CV_results.csv'))

def pickle_model(file_name, model, cv_results):
    joblib.dump(model, os.path.join('models', file_name+'.pkl'))
    pickle_cv_results(file_name, cv_results)
    print(file_name)

print("✅ helper functions ready")


In [ ]:

# =========================================================
# 2) Excel → (저자 코드가 기대하는) CSV 생성
# =========================================================

xlsx_path = DATA_XLSX_PATH
if not os.path.exists(xlsx_path):
    if os.path.exists(FALLBACK_XLSX_PATH):
        print("⚠️ DATA_XLSX_PATH not found, using fallback:", FALLBACK_XLSX_PATH)
        xlsx_path = FALLBACK_XLSX_PATH
    else:
        raise FileNotFoundError(f"Excel not found:\n- {DATA_XLSX_PATH}\n- {FALLBACK_XLSX_PATH}")

chem_df = pd.read_excel(xlsx_path, sheet_name=CHEM_SHEET)
sens_df = pd.read_excel(xlsx_path, sheet_name=SENS_SHEET)

# 저자 노트북 파일명과 동일하게 저장(현재 RUN_DIR 안)
chem_csv = "Supplemental File S1.csv"
sens_csv = "Supplemental File S4.csv"

chem_df.to_csv(chem_csv, index=False)
sens_df.to_csv(sens_csv, index=False)

print("✅ wrote:", chem_csv, "shape=", chem_df.shape)
print("✅ wrote:", sens_csv, "shape=", sens_df.shape)


In [ ]:

# =========================================================
# 3) 저자 코드: 데이터 로드 / split / scaling / interactions
# =========================================================

chem_dataset = pd.read_csv('Supplemental File S1.csv').set_index('beer')
trained_panel_dataset = pd.read_csv('Supplemental File S4.csv').set_index('beer')

chem_dataset_expertpanel = chem_dataset.join(trained_panel_dataset)

x_name = 'chem_log'
X = generate_X(pd.concat([chem_dataset_expertpanel['beer_id'] ,
                          chem_dataset_expertpanel.loc[:,'acetaldehyde':'sulfur_sum']], axis=1),
               impute=True)

y_name = 'expertpanel'
Y = chem_dataset_expertpanel.loc[:,'A_malt_all':'overall']
y_class = chem_dataset_expertpanel.loc[:,'tasting_category_fine']

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.30,
    random_state=0,
    shuffle=True,
    stratify=y_class
)

# Scaling X (저자 노트북 그대로)
scaler_X = StandardScaler().fit(X_train)
X_train = scaler_X.transform(X_train)
X_test  = scaler_X.transform(X_test)

# Creating first order interaction terms (Linear/Lasso 용)
poly = PolynomialFeatures(interaction_only=True)
X_train_int = poly.fit_transform(X_train)
X_test_int  = poly.fit_transform(X_test)

X_train_int = pd.DataFrame(X_train_int, columns=poly.get_feature_names_out()).drop('1', axis=1)
X_test_int  = pd.DataFrame(X_test_int,  columns=poly.get_feature_names_out()).drop('1', axis=1)

print("X_train:", X_train.shape, "X_train_int:", X_train_int.shape)
print("Y_train:", Y_train.shape)


In [ ]:

# =========================================================
# 4) Linear model (interaction terms)
# =========================================================
if RUN_LINEAR_LASSO:
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        regressor_loop = LinearRegression(fit_intercept=True, copy_X=True)
        regressor_loop.fit(X_train_int, Y_train_loop)

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train_int, Y_train_loop, X_test_int, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['LinearModel interactions', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        joblib.dump(regressor_loop, os.path.join('models', f'Linear_regression_interaction_{aspect}_{y_name}.pkl'))

    # Training full model
    LM = LinearRegression(fit_intercept=True, copy_X=True).fit(X_train_int, Y_train)
    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train_int, Y_train, X_test_int, Y_test, LM, toprint=False
    )
    aspect = 'everything'
    joblib.dump(LM, os.path.join('models', f'Linear_regression_interaction_full_model_{y_name}.pkl'))
    performance_loop.append(['Complete_LinearModel interactions', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_lr_df = pd.DataFrame(performance_loop,
                                     columns=['regressor','X', 'Y', 'aspect',
                                              'mse_train', 'r2_train', 'r_train',
                                              'mse_test', 'r2_test', 'r_test'])
    performance_lr_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_Linear_model_interactions.csv',
        index=False
    )
    print("✅ saved Linear performance CSV")

    # =========================================================
    # 5) Lasso regression (interaction terms)
    # =========================================================
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        regressor_loop = LassoCV(cv=5, random_state=0)
        regressor_loop.fit(X_train_int, Y_train_loop)

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train_int, Y_train_loop, X_test_int, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['Lasso interactions', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        joblib.dump(regressor_loop, os.path.join('models', f'Lasso_regression_interactions_{aspect}_{y_name}.pkl'))

    # Training full model
    lasso_cv = MultiTaskLassoCV(cv=5, random_state=0).fit(X_train_int, Y_train)
    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train_int, Y_train, X_test_int, Y_test, lasso_cv, toprint=False
    )
    aspect = 'everything'
    joblib.dump(lasso_cv, os.path.join('models', f'Lasso_regression_interactions_full_model_{y_name}.pkl'))
    performance_loop.append(['Complete_Lasso interactions', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_l1r_df = pd.DataFrame(performance_loop,
                                      columns=['regressor','X', 'Y', 'aspect',
                                               'mse_train', 'r2_train', 'r_train',
                                               'mse_test', 'r2_test', 'r_test'])
    performance_l1r_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_Lasso_interactions.csv',
        index=False
    )
    print("✅ saved Lasso performance CSV")
else:
    print("SKIP: RUN_LINEAR_LASSO=False")


In [ ]:

# =========================================================
# 6) Partial Least Squares Regression
# =========================================================
if RUN_PLS:
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        regressor_loop = PLSRegression(scale=True)
        regressor_loop.fit_transform(X_train, Y_train_loop)

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train, Y_train_loop, X_test, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['PLS', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        joblib.dump(regressor_loop, os.path.join('models', f'PLS_regression_{aspect}_{y_name}.pkl'))

    # Training full model
    PLS = PLSRegression().fit(X_train, Y_train)
    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train, Y_train, X_test, Y_test, PLS, toprint=False
    )
    aspect = 'everything'

    # (저자 노트북의 파일명 오타를 유지)
    joblib.dump(PLS, os.path.join('models', f'Elasticnet_regression_full_model_{y_name}.pkl'))

    performance_loop.append(['Complete_PLS', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_pls_df = pd.DataFrame(performance_loop,
                                      columns=['regressor','X', 'Y', 'aspect',
                                               'mse_train', 'r2_train', 'r_train',
                                               'mse_test', 'r2_test', 'r_test'])
    performance_pls_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_Partial_Least_Squares_Regression.csv',
        index=False
    )
    print("✅ saved PLS performance CSV")
else:
    print("SKIP: RUN_PLS=False")


In [ ]:

# =========================================================
# 7) AdaBoost
# =========================================================
if RUN_ADABOOST:
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        param_grid = [{'learning_rate': np.arange(0.2, 2.0, 0.2),
                       'loss': ['linear','square','exponential'],
                       'n_estimators': [200, 500, 1000]}]

        grid_search_loop = GridSearchCV(
            AdaBoostRegressor(n_estimators=200, random_state=0),
            param_grid,
            cv=5,
            scoring='r2',
            verbose=1,
            refit=True,
            n_jobs=N_JOBS
        )
        grid_search_results = grid_search_loop.fit(X_train, Y_train_loop)
        regressor_loop = grid_search_results.best_estimator_

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train, Y_train_loop, X_test, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['AdaBoost', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        CV_results = pd.DataFrame(grid_search_loop.cv_results_)
        pickle_model(f'Adaboost_regressor_{aspect}_{y_name}', regressor_loop, CV_results)

    # Full model
    param_grid = [{'estimator__learning_rate': np.arange(0.2, 2.0, 0.2),
                   'estimator__loss': ['linear','square','exponential'],
                   'estimator__n_estimators': [200, 500, 1000]}]
    grid_search = GridSearchCV(
        MultiOutputRegressor(AdaBoostRegressor(random_state=0)),
        param_grid,
        cv=5,
        scoring='r2',
        verbose=1,
        refit=True,
        n_jobs=N_JOBS
    )
    grid_search_results = grid_search.fit(X_train, Y_train)
    adaboost = grid_search_results.best_estimator_

    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train, Y_train, X_test, Y_test, adaboost, toprint=False
    )
    aspect = 'everything'

    CV_results = pd.DataFrame(grid_search.cv_results_)
    pickle_model(f'Adaboost_regressor_full_model_{y_name}', adaboost, CV_results)

    performance_loop.append(['Complete_Adaboost', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_abr_df = pd.DataFrame(performance_loop,
                                      columns=['regressor','X', 'Y', 'aspect',
                                               'mse_train', 'r2_train', 'r_train',
                                               'mse_test', 'r2_test', 'r_test'])
    performance_abr_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_AdaBoost.csv',
        index=False
    )
    print("✅ saved AdaBoost performance CSV")
else:
    print("SKIP: RUN_ADABOOST=False")


In [ ]:

# =========================================================
# 8) GradientBoostingRegressor
# =========================================================
if RUN_GRADBOOST:
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        param_grid = [{
            'subsample': np.arange(0.2, 1.0, 0.2),
            'learning_rate': np.arange(0.02, 0.2, 0.02),
            'max_depth': np.arange(6, 15, 3),
            'n_estimators': [200, 500, 1000],
            'loss': ['squared_error', 'absolute_error']
        }]

        grid_search_loop = GridSearchCV(
            GradientBoostingRegressor(random_state=0),
            param_grid,
            cv=5,
            scoring='r2',
            verbose=1,
            refit=True,
            return_train_score=True,
            n_jobs=N_JOBS
        )
        grid_search_results = grid_search_loop.fit(X_train, Y_train_loop)
        regressor_loop = grid_search_results.best_estimator_

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train, Y_train_loop, X_test, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['GradientBoost', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        CV_results = pd.DataFrame(grid_search_loop.cv_results_)
        pickle_model(f'Gradient_Boost_regressor_{aspect}_{y_name}', regressor_loop, CV_results)

    # Full model
    param_grid = [{
        'estimator__subsample': np.arange(0.2, 1.0, 0.2),
        'estimator__learning_rate': np.arange(0.02, 0.2, 0.02),
        'estimator__max_depth': np.arange(6, 15, 3),
        'estimator__n_estimators': [200, 500, 1000],
        'estimator__loss': ['squared_error', 'absolute_error']
    }]
    grid_search = GridSearchCV(
        MultiOutputRegressor(GradientBoostingRegressor(random_state=0)),
        param_grid,
        cv=5,
        scoring='r2',
        verbose=1,
        refit=True,
        return_train_score=True,
        n_jobs=N_JOBS
    )
    grid_search_results = grid_search.fit(X_train, Y_train)
    gradientboost = grid_search_results.best_estimator_

    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train, Y_train, X_test, Y_test, gradientboost, toprint=False
    )
    aspect = 'everything'

    CV_results = pd.DataFrame(grid_search.cv_results_)
    pickle_model(f'Gradient_Boost_regressor_full_model_{y_name}', gradientboost, CV_results)

    performance_loop.append(['Complete_GradientBoost', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_gb_df = pd.DataFrame(performance_loop,
                                     columns=['regressor','X', 'Y', 'aspect',
                                              'mse_train', 'r2_train', 'r_train',
                                              'mse_test', 'r2_test', 'r_test'])
    performance_gb_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_GradientBoost.csv',
        index=False
    )
    print("✅ saved GradientBoost performance CSV")
else:
    print("SKIP: RUN_GRADBOOST=False")


In [ ]:

# =========================================================
# 9) Random Forest
# =========================================================
if RUN_RF:
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        param_grid = [{
            'max_features': np.arange(2, X_train.shape[1], 20),
            'max_depth': np.arange(4, 20, 2),
            'min_impurity_decrease': np.arange(0, 0.4, 0.05),
            'n_estimators': [200, 500, 1000]
        }]

        grid_search_loop = GridSearchCV(
            estimator=RandomForestRegressor(oob_score=True, random_state=0),
            param_grid=param_grid,
            cv=5,
            scoring='r2',
            refit=True,
            verbose=1,
            return_train_score=True,
            n_jobs=N_JOBS
        )
        grid_search_results = grid_search_loop.fit(X_train, Y_train_loop)
        regressor_loop = grid_search_results.best_estimator_

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train, Y_train_loop, X_test, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['RandomForest', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        CV_results = pd.DataFrame(grid_search_loop.cv_results_)
        pickle_model(f'Random_Forest_regressor_{aspect}_{y_name}', regressor_loop, CV_results)

    # Full model
    param_grid = [{
        'max_features': np.arange(2, X_train.shape[1], 20),
        'max_depth': np.arange(4, 20, 2),
        'min_impurity_decrease': np.arange(0, 0.4, 0.05),
        'n_estimators': [200, 500, 1000]
    }]

    grid_search = GridSearchCV(
        estimator=RandomForestRegressor(oob_score=True, random_state=0),
        param_grid=param_grid,
        cv=5,
        scoring='r2',
        refit=True,
        verbose=1,
        return_train_score=True,
        n_jobs=N_JOBS
    )
    grid_search_results = grid_search.fit(X_train, Y_train)
    random_forest = grid_search_results.best_estimator_

    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train, Y_train, X_test, Y_test, random_forest, toprint=False
    )
    aspect = 'everything'

    CV_results = pd.DataFrame(grid_search.cv_results_)
    pickle_model(f'Random_Forest_regressor_full_model_{y_name}', random_forest, CV_results)

    performance_loop.append(['Complete_RandomForest', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_rf_df = pd.DataFrame(performance_loop,
                                     columns=['regressor','X', 'Y', 'aspect',
                                              'mse_train', 'r2_train', 'r_train',
                                              'mse_test', 'r2_test', 'r_test'])
    performance_rf_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_RandomForest.csv',
        index=False
    )
    print("✅ saved RandomForest performance CSV")
else:
    print("SKIP: RUN_RF=False")


In [ ]:

# =========================================================
# 10) Extra Trees
# =========================================================
if RUN_EXTRATREES:
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        param_grid = [{
            'max_features': np.arange(2, X_train.shape[1], 20),
            'max_depth': np.arange(4, 20, 2),
            'min_impurity_decrease': np.arange(0, 0.4, 0.05),
            'n_estimators': [200, 500, 1000]
        }]

        grid_search_loop = GridSearchCV(
            estimator=ExtraTreesRegressor(bootstrap=True, oob_score=True, random_state=0),
            param_grid=param_grid,
            cv=5,
            scoring='r2',
            refit=True,
            verbose=1,
            n_jobs=N_JOBS
        )
        grid_search_results = grid_search_loop.fit(X_train, Y_train_loop)
        regressor_loop = grid_search_results.best_estimator_

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train, Y_train_loop, X_test, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['ExtraTrees', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        CV_results = pd.DataFrame(grid_search_loop.cv_results_)
        pickle_model(f'Extra_Trees_regressor_{aspect}_{y_name}', regressor_loop, CV_results)

    # Full model
    param_grid = [{
        'max_features': np.arange(2, X_train.shape[1], 20),
        'max_depth': np.arange(4, 20, 2),
        'min_impurity_decrease': np.arange(0, 0.4, 0.05),
        'n_estimators': [200, 500, 1000]
    }]

    grid_search = GridSearchCV(
        estimator=ExtraTreesRegressor(bootstrap=True, oob_score=True, random_state=0),
        param_grid=param_grid,
        cv=5,
        scoring='r2',
        refit=True,
        verbose=1,
        n_jobs=N_JOBS
    )
    grid_search_results = grid_search.fit(X_train, Y_train)
    extra_trees = grid_search_results.best_estimator_

    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train, Y_train, X_test, Y_test, extra_trees, toprint=False
    )
    aspect = 'everything'

    CV_results = pd.DataFrame(grid_search.cv_results_)
    pickle_model(f'Extra_Trees_regressor_full_model_{y_name}', extra_trees, CV_results)

    performance_loop.append(['Complete_ExtraTrees', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_et_df = pd.DataFrame(performance_loop,
                                     columns=['regressor','X', 'Y', 'aspect',
                                              'mse_train', 'r2_train', 'r_train',
                                              'mse_test', 'r2_test', 'r_test'])
    performance_et_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_ExtraTrees.csv',
        index=False
    )
    print("✅ saved ExtraTrees performance CSV")
else:
    print("SKIP: RUN_EXTRATREES=False")


In [ ]:

# =========================================================
# 11) XGBoost
# =========================================================
if RUN_XGBOOST:
    if XGBRegressor is None:
        raise ImportError("xgboost가 설치되어 있지 않아 XGBoost 섹션을 실행할 수 없습니다.")

    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        param_grid = [{
            'eta': [0.001, 0.005, 0.01, 0.05, 0.1, 0.3, 0.5],
            'max_depth': np.arange(10) + 1,
            'n_estimators': [200, 500, 1000],
        }]

        grid_search_loop = GridSearchCV(
            estimator=XGBRegressor(oob_score=True, random_state=0),
            param_grid=param_grid,
            cv=5,
            scoring='r2',
            refit=True,
            verbose=1,
            return_train_score=True,
            n_jobs=N_JOBS
        )
        grid_search_results = grid_search_loop.fit(X_train, Y_train_loop)
        regressor_loop = grid_search_results.best_estimator_

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train, Y_train_loop, X_test, Y_test_loop, regressor_loop, toprint=False
        )

        # (저자 노트북의 레이블 오타를 유지: 'RandomForest')
        performance_loop.append(['RandomForest', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        CV_results = pd.DataFrame(grid_search_loop.cv_results_)
        pickle_model(f'XGBoost_regressor_{aspect}_{y_name}', regressor_loop, CV_results)

    # Full model
    param_grid = [{
        'estimator__eta': [0.001, 0.005, 0.01, 0.05, 0.1, 0.3, 0.5],
        'estimator__max_depth': np.arange(10) + 1,
        'estimator__n_estimators': [200, 500, 1000],
    }]

    grid_search = GridSearchCV(
        estimator=MultiOutputRegressor(XGBRegressor(random_state=0)),
        param_grid=param_grid,
        cv=5,
        scoring='r2',
        refit=True,
        verbose=1,
        return_train_score=True,
        n_jobs=N_JOBS
    )
    grid_search_results = grid_search.fit(X_train, Y_train)
    XGBoost = grid_search_results.best_estimator_

    # (저자 노트북의 성능평가 오타: random_forest 사용)
    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train, Y_train, X_test, Y_test, random_forest, toprint=False
    )
    aspect = 'everything'

    CV_results = pd.DataFrame(grid_search.cv_results_)
    pickle_model(f'XGBoost_regressor_full_model_{y_name}', XGBoost, CV_results)

    performance_loop.append(['Complete_XGBoost', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_xgb_df = pd.DataFrame(performance_loop,
                                      columns=['regressor','X', 'Y', 'aspect',
                                               'mse_train', 'r2_train', 'r_train',
                                               'mse_test', 'r2_test', 'r_test'])
    performance_xgb_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_XGBoost.csv',
        index=False
    )
    print("✅ saved XGBoost performance CSV")
else:
    print("SKIP: RUN_XGBOOST=False")


In [ ]:

# =========================================================
# 12) Support Vector Regressor (GridSearch가 매우 큼)
# =========================================================
from sklearn.svm import SVR

if RUN_SVR:
    performance_loop = []
    for aspect in Y_train.columns:
        Y_train_loop = Y_train[aspect]
        Y_test_loop  = Y_test[aspect]

        param_grid = [{
            'C': [0.001, 0.01, 0.1, 1, 10],
            'epsilon': [0.01, 0.05, 0.1, 0.2],
            'kernel': ['linear','poly','rbf'],
            'degree': np.arange(2,5),
            'gamma': np.arange(0.002,0.5,0.01)
        }]

        grid_search_loop = GridSearchCV(
            SVR(),
            param_grid,
            cv=5,
            scoring='r2',
            verbose=1,
            refit=True,
            n_jobs=N_JOBS
        )
        grid_search_results = grid_search_loop.fit(X_train, Y_train_loop)
        regressor_loop = grid_search_results.best_estimator_

        mse_train, r2_train, mse_test, r2_test = regressor_performance(
            X_train, Y_train_loop, X_test, Y_test_loop, regressor_loop, toprint=False
        )
        performance_loop.append(['SupportVectorMachine', x_name, y_name, aspect,
                                 mse_train, r2_train, np.sqrt(r2_train),
                                 mse_test,  r2_test,  np.sqrt(r2_test)])

        CV_results = pd.DataFrame(grid_search_loop.cv_results_)
        pickle_model(f'Support_Vector_regressor_{aspect}_{y_name}', regressor_loop, CV_results)

    # Full model
    param_grid = [{
        'estimator__C': [0.001, 0.01, 0.1, 1, 10],
        'estimator__epsilon': [0.01, 0.05, 0.1, 0.2],
        'estimator__kernel': ['linear','poly','rbf'],
        'estimator__degree': np.arange(2,5),
        'estimator__gamma': np.arange(0.002,0.5,0.01)
    }]

    grid_search = GridSearchCV(
        MultiOutputRegressor(SVR()),
        param_grid,
        cv=5,
        scoring='r2',
        verbose=1,
        refit=True,
        n_jobs=N_JOBS
    )
    grid_search_results = grid_search.fit(X_train, Y_train)
    svm = grid_search_results.best_estimator_

    mse_train, r2_train, mse_test, r2_test = regressor_performance(
        X_train, Y_train, X_test, Y_test, svm, toprint=False
    )
    aspect = 'everything'

    CV_results = pd.DataFrame(grid_search.cv_results_)
    pickle_model(f'Support_Vector_regressor_full_model_{y_name}', svm, CV_results)

    performance_loop.append(['Complete_SupportVectorMachine', x_name, y_name, aspect,
                             mse_train, r2_train, np.sqrt(r2_train),
                             mse_test,  r2_test,  np.sqrt(r2_test)])

    performance_svm_df = pd.DataFrame(performance_loop,
                                      columns=['regressor','X', 'Y', 'aspect',
                                               'mse_train', 'r2_train', 'r_train',
                                               'mse_test', 'r2_test', 'r_test'])
    performance_svm_df.to_csv(
        f'Model_performance/ModelPerformances_{x_name}_{y_name}_SupportVectorMachine.csv',
        index=False
    )
    print("✅ saved SVR performance CSV")
else:
    print("SKIP: RUN_SVR=False")


In [ ]:

# =========================================================
# 13) MLPRegressor (Optuna) — 저자 노트북 방식(random_state=1 split)
# =========================================================
from sklearn.neural_network import MLPRegressor

if RUN_MLP_OPTUNA:
    if optuna is None:
        raise ImportError("optuna가 설치되어 있지 않아 MLP(Optuna) 섹션을 실행할 수 없습니다.")

    # 저자 노트북은 별도 파일을 로드하지만, 여기서는 동일 데이터(chem+sens join)를 런폴더에 저장 후 로드
    chem_dataset_complete = chem_dataset.join(trained_panel_dataset)
    chem_dataset_complete.to_csv('chem_dataset_complete_log_expertpanel.csv')

    chem_dataset_expertpanel_nn = pd.read_csv('chem_dataset_complete_log_expertpanel.csv').set_index('beer')

    x_name = 'chem_log'
    X = generate_X(pd.concat([chem_dataset_expertpanel_nn['beer_id'],
                              chem_dataset_expertpanel_nn.loc[:,'acetaldehyde':'sulfur_sum']], axis=1),
                   impute=True)
    y_name = 'expertpanel'
    Y = chem_dataset_expertpanel_nn.loc[:,'A_malt_all':'overall']
    y_class = chem_dataset_expertpanel_nn.loc[:,'tasting_category_fine']

    X_train, X_test, Y_train, Y_test = train_test_split(
        X, Y,
        test_size=0.30,
        random_state=1,
        shuffle=True,
        stratify=y_class
    )

    Model_performance = []

    # ---- individual models ----
    for aspect in Y_train.columns:

        def objective(trial):
            n_layers = trial.suggest_int('n_layers', 1, 10)
            layers = []
            for i in range(n_layers):
                layers.append(trial.suggest_int(f'n_units_{i+1}', 20, 300, step=40))
            activation = trial.suggest_categorical('activation', ['tanh','relu'])
            alpha = trial.suggest_float('alpha', 1e-4, 100, log=True)

            model = MLPRegressor(
                random_state=1,
                solver='adam',
                activation=activation,
                alpha=alpha,
                hidden_layer_sizes=(layers),
                max_iter=1000,
                learning_rate='adaptive',
                early_stopping=True
            )

            scores = cross_val_score(
                model,
                X_train,
                Y_train[aspect],
                cv=KFold(n_splits=5, shuffle=False),
                scoring='r2',
                n_jobs=N_JOBS
            )
            return scores.mean()

        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=MLP_N_TRIALS, n_jobs=N_JOBS)

        architecture = []
        for i in range(study.best_params['n_layers']):
            architecture.append(study.best_params[f'n_units_{i+1}'])

        model = MLPRegressor(
            random_state=1,
            solver='adam',
            max_iter=1000,
            learning_rate='adaptive',
            early_stopping=True,
            hidden_layer_sizes=architecture,
            activation=study.best_params['activation'],
            alpha=study.best_params['alpha']
        )

        model.fit(X_train, Y_train[aspect])
        joblib.dump(model, os.path.join('models', f'Optuna_Neural_network_model_adam_{y_name}_{aspect}.pkl'))

        myresults = study.best_params
        myresults['R2'] = regressor_performance(X_train, Y_train[aspect], X_test, Y_test[aspect], model, toprint=False)[3]
        mydf = pd.DataFrame.from_dict(myresults, orient='index')
        mydf.columns = ['optimal value']
        mydf.to_csv(f'Optimal_settings/Optimal_settings_neural_network_adam_{y_name}_{aspect}.csv')

        mse_train, r2_train, mse_test, r2_test = regressor_performance(X_train, Y_train[aspect], X_test, Y_test[aspect], model, toprint=False)
        Model_performance.append(['MLPRegressor', x_name, y_name, aspect,
                                  mse_train, r2_train, np.sqrt(r2_train),
                                  mse_test,  r2_test,  np.sqrt(r2_test)])

    # ---- full model ----
    def objective(trial):
        n_layers = trial.suggest_int('n_layers', 1, 10)
        layers = []
        for i in range(n_layers):
            layers.append(trial.suggest_int(f'n_units_{i+1}', 20, 300, step=40))
        activation = trial.suggest_categorical('activation', ['tanh','relu'])
        alpha = trial.suggest_float('alpha', 1e-4, 100, log=True)

        model = MLPRegressor(
            random_state=1,
            solver='adam',
            activation=activation,
            alpha=alpha,
            hidden_layer_sizes=(layers),
            max_iter=1000,
            learning_rate='adaptive',
            early_stopping=True
        )

        scores = cross_val_score(
            model,
            X_train,
            Y_train,
            cv=KFold(n_splits=5, shuffle=False),
            scoring='r2',
            n_jobs=N_JOBS
        )
        return scores.mean()

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=MLP_N_TRIALS, n_jobs=N_JOBS)

    architecture = []
    for i in range(study.best_params['n_layers']):
        architecture.append(study.best_params[f'n_units_{i+1}'])

    model = MLPRegressor(
        random_state=1,
        solver='adam',
        max_iter=1000,
        learning_rate='adaptive',
        early_stopping=True,
        hidden_layer_sizes=architecture,
        activation=study.best_params['activation'],
        alpha=study.best_params['alpha']
    )

    model.fit(X_train, Y_train)
    joblib.dump(model, os.path.join('models', f'Optuna_Neural_network_full_model_adam_{y_name}.pkl'))

    myresults = study.best_params
    myresults['R2'] = regressor_performance(X_train, Y_train, X_test, Y_test, model, toprint=False)[3]
    mydf = pd.DataFrame.from_dict(myresults, orient='index')
    mydf.columns = ['optimal value']
    mydf.to_csv(f'Optimal_settings/Optimal_settings_neural_network_adam_full_model_{y_name}.csv')

    mse_train, r2_train, mse_test, r2_test = regressor_performance(X_train, Y_train, X_test, Y_test, model, toprint=False)
    Model_performance.append(['Complete_MLPRegressor', x_name, y_name, 'everything',
                              mse_train, r2_train, np.sqrt(r2_train),
                              mse_test,  r2_test,  np.sqrt(r2_test)])

    Performance = pd.DataFrame(Model_performance,
                               columns=['regressor','X', 'Y', 'aspect',
                                        'mse_train', 'r2_train', 'r_train',
                                        'mse_test', 'r2_test', 'r_test'])

    # (저자 노트북의 경로 오타를 유지: Model_Performance)
    Performance.to_csv(f'Model_Performance/ModelPerformances_{x_name}_{y_name}_MLPRegressor.csv', index=False)
    print("✅ saved MLP performance CSV")

else:
    print("SKIP: RUN_MLP_OPTUNA=False")


In [ ]:

# =========================================================
# 14) 최종 요약: 생성된 결과 파일 빠르게 보기
# =========================================================
import glob

print("RUN_DIR:", os.getcwd())

print("\n[Model_performance CSVs]")
for p in sorted(glob.glob("Model_performance/*.csv")):
    print(" -", p)

print("\n[Model_Performance CSVs]")
for p in sorted(glob.glob("Model_Performance/*.csv")):
    print(" -", p)

print("\n[Models]")
print(" - #models:", len(glob.glob("models/*.pkl")))

print("\n[CV_results]")
print(" - #cv csv:", len(glob.glob("CV_results/*_CV_results.csv")))
